In [ ]:
# Run from the repo root so relative imports and `!`-shell commands resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# Sample image visualization

Plain `clean / poisoned / noise` triplets (no GradCAM). For each sample one row:

| Clean $x$ | Poisoned $x{+}\delta$ | Noise $\delta$ |

Set `TRANSPOSE = True` to switch to *rows = type, cols = samples* (wider figure).
Adjust `DELTA_VIS_SCALE` (1.0 = same scale as image, ≥4 = amplified for visibility).

In [ ]:
import os
import numpy as np
import torch
from torchvision import transforms
import mxnet as mx
import matplotlib.pyplot as plt

from backbones.custom_generator import ResNetGenerator

## Config

In [ ]:
# TODO: local paths in this cell were redacted to ""; set them before running.
GEN_CKPT          = ""
CKPT_CLEAN        = ""     # R-50 trained on D_c
CKPT_POISON       = ""    # R-50 trained on D_u

TEST_SOURCE       = 'lfw'
DATA_ROOT_CASIA   = ""
LFW_BIN           = "" # contains train.rec / train.idx

N_SAMPLES         = 5
TRANSPOSE         = True         # True: rows=type, cols=samples | False: rows=samples, cols=type
DELTA_VIS_SCALE   = 1.0          # 1.0 = same scale as image (almost gray); 4–10 = amplified

OUT_PATH          = "sample_imgs.pdf"
SEED              = 8
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEVICE,
      ' N:', N_SAMPLES,
      ' transpose:', TRANSPOSE,
      ' delta scale:', DELTA_VIS_SCALE)

## Sample loader

In [ ]:
class MXFaceDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir):
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])
        self.imgrec = mx.recordio.MXIndexedRecordIO(
            os.path.join(root_dir, 'train.idx'),
            os.path.join(root_dir, 'train.rec'), 'r',
        )
        s = self.imgrec.read_idx(0)
        header, _ = mx.recordio.unpack(s)
        if header.flag > 0:
            self.imgidx = np.array(range(1, int(header.label[0])))
        else:
            self.imgidx = np.array(list(self.imgrec.keys))

    def __len__(self):
        return len(self.imgidx)

    def __getitem__(self, index):
        idx = self.imgidx[index]
        s = self.imgrec.read_idx(int(idx))
        _, img_bytes = mx.recordio.unpack(s)
        img = mx.image.imdecode(img_bytes).asnumpy()
        return self.transform(img)


def sample_inputs(n):
    ds = MXFaceDataset(DATA_ROOT_CASIA)
    idx = np.random.permutation(len(ds))[:n]
    return torch.stack([ds[int(i)] for i in idx]).to(DEVICE)

## Load generator + build clean / poisoned / δ

In [ ]:
G = ResNetGenerator().to(DEVICE).eval()
state = torch.load(GEN_CKPT, map_location=DEVICE)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
state = {k.replace('module.', ''): v for k, v in state.items()}
G.load_state_dict(state, strict=False)

x = sample_inputs(N_SAMPLES)
with torch.no_grad():
    delta = G(x)
    poison = (x + delta).clamp(-1, 1)
print('clean:', x.shape, ' poisoned:', poison.shape, ' delta:', delta.shape)

## Visualize

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def img_to_disp(t):
    a = t.detach().cpu().numpy()
    return np.clip(((a + 1.0) / 2.0).transpose(1, 2, 0), 0, 1)

def delta_to_disp(t, scale=DELTA_VIS_SCALE):
    a = t.detach().cpu().numpy() * scale
    return np.clip(((a + 1.0) / 2.0).transpose(1, 2, 0), 0, 1)

TYPES = ['clean', 'poisoned', 'delta']
TITLES = {
    'clean':    r'Clean $x$',
    'poisoned': r'Poisoned $x{+}\delta$',
    'delta':    rf'Noise $\delta$ (×{int(DELTA_VIS_SCALE)})' if DELTA_VIS_SCALE != 1.0 else r'Noise $\delta$',
}
TENSORS = {'clean': x, 'poisoned': poison, 'delta': delta}

def cell_disp(type_key, idx):
    t = TENSORS[type_key][idx]
    return delta_to_disp(t) if type_key == 'delta' else img_to_disp(t)

N = N_SAMPLES
cell = 1.0
if not TRANSPOSE:
    n_rows, n_cols = N, 3
else:
    n_rows, n_cols = 3, N

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(cell * n_cols, cell * n_rows),
                         dpi=200, squeeze=False)

for r in range(n_rows):
    for c in range(n_cols):
        if not TRANSPOSE:
            sample_idx, type_key = r, TYPES[c]
            if r == 0:
                axes[r, c].set_title(TITLES[type_key], fontsize=8, pad=2)
        else:
            sample_idx, type_key = c, TYPES[r]
            if c == 0:
                axes[r, c].set_ylabel(TITLES[type_key], fontsize=8)
        axes[r, c].imshow(cell_disp(type_key, sample_idx))
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        for s in axes[r, c].spines.values():
            s.set_linewidth(0.4)

plt.tight_layout(pad=0.3, h_pad=0.1, w_pad=0.2)
fig.savefig(OUT_PATH, bbox_inches='tight')
fig.savefig(OUT_PATH.replace('.pdf', '.png'), bbox_inches='tight', dpi=200)
plt.show()
print('Saved', OUT_PATH)

## Quality metrics over the full LFW set

For every image in `LFW_BIN` (InsightFace pair-test format: pickled `(bins, issame_list)`):
push it through the generator $G$, compute the perturbation $\delta = G(x)$,
and report **PSNR / SSIM / $\ell_\infty$** as mean ± std.

- PSNR / SSIM are computed in the conventional [0, 1] image scale.
- $\ell_\infty$ is reported in 0–255 scale (the usual $\varepsilon$ convention).

In [ ]:
from eval.verification import load_bin
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **k): return it

# --- Load LFW with InsightFace's standard loader (matches Bench evaluation) ---
# Returns: data_list = [orig, h-flipped], each [N, 3, 112, 112] in [0, 255] float;
#          issame_list of length N/2.
lfw_data, issame_list = load_bin(LFW_BIN, (112, 112))
lfw_imgs = lfw_data[0]                # use originals (non-flipped) for metric eval
N_TOTAL = lfw_imgs.size(0)
print(f'LFW images: {N_TOTAL}  ({len(issame_list)} pairs)')

BATCH = 64
psnrs, ssims, linfs = [], [], []

G.eval()
with torch.no_grad():
    for i in tqdm(range(0, N_TOTAL, BATCH), desc='LFW metrics'):
        # load_bin gives float in [0, 255] → convert to [-1, 1] for the generator
        x_raw   = lfw_imgs[i:i + BATCH].to(DEVICE)
        x_batch = (x_raw - 127.5) / 127.5
        d_batch = G(x_batch)
        p_batch = (x_batch + d_batch).clamp(-1, 1)

        # [0, 1] for standard metric computation
        x01 = (x_batch + 1) / 2
        p01 = (p_batch + 1) / 2

        # ℓ∞ per image, reported in 0-255 scale
        linf = (p01 - x01).abs().amax(dim=[1, 2, 3]) * 255.0
        linfs.append(linf.cpu().numpy())

        # PSNR / SSIM per image (skimage on CPU)
        x_np = x01.cpu().numpy().transpose(0, 2, 3, 1)
        p_np = p01.cpu().numpy().transpose(0, 2, 3, 1)
        for j in range(x_np.shape[0]):
            psnrs.append(psnr_fn(x_np[j], p_np[j], data_range=1.0))
            ssims.append(ssim_fn(x_np[j], p_np[j], data_range=1.0, channel_axis=-1))

psnrs = np.array(psnrs)
linfs = np.concatenate(linfs)
ssims = np.array(ssims)

print(f'\n[LFW full set | N={len(psnrs)}]')
print(f'  PSNR (dB)     : {psnrs.mean():7.3f} ± {psnrs.std():.3f}')
print(f'  SSIM          : {ssims.mean():7.4f} ± {ssims.std():.4f}')
print(f'  L_inf (0-255) : {linfs.mean():7.3f} ± {linfs.std():.3f}')

In [ ]:
psnrs, ssims, linfs = [], [], []

G.eval()
with torch.no_grad():
    for i in tqdm(range(0, N_TOTAL, BATCH), desc='LFW metrics'):
        # load_bin gives float in [0, 255] → convert to [-1, 1] for the generator
        x_raw   = lfw_imgs[i:i + BATCH].to(DEVICE)
        x_batch = (x_raw - 127.5) / 127.5
        d_batch = G(x_batch)
        p_batch = (x_batch + d_batch).clamp(-1, 1)

        # [0, 1] for standard metric computation
        x01 = (x_batch + 1) / 2
        p01 = (p_batch + 1) / 2

        # ℓ∞ per image, reported in 0-255 scale
        linf = (p01 - x01).abs().amax(dim=[1, 2, 3]) * 255.0
        linfs.append(linf.cpu().numpy())

        # PSNR / SSIM per image (skimage on CPU)
        x_np = x01.cpu().numpy().transpose(0, 2, 3, 1)
        p_np = p01.cpu().numpy().transpose(0, 2, 3, 1)
        for j in range(x_np.shape[0]):
            psnrs.append(psnr_fn(x_np[j], p_np[j], data_range=1.0))
            ssims.append(ssim_fn(x_np[j], p_np[j], data_range=1.0, channel_axis=-1,
                                gaussian_weights=True,
                                sigma = 1.5,
                                use_sample_covariance=False))

psnrs = np.array(psnrs)
linfs = np.concatenate(linfs)
ssims = np.array(ssims)

print(f'\n[LFW full set | N={len(psnrs)}]')
print(f'  PSNR (dB)     : {psnrs.mean():7.3f} ± {psnrs.std():.3f}')
print(f'  SSIM          : {ssims.mean():7.4f} ± {ssims.std():.4f}')
print(f'  L_inf (0-255) : {linfs.mean():7.3f} ± {linfs.std():.3f}')

## SSIM sanity check: baselines on the same LFW set

Computes SSIM/PSNR for *trivial* perturbations applied to the same LFW images, to see whether 0.87 is a property of our generator or just what 112×112 aligned faces look like under any ε=8/255 perturbation:

- **Random uniform** `[-ε, +ε]` per pixel — typical "non-structured" noise of same budget
- **Random sign** `±ε` (Bernoulli) — worst-case for fixed $\ell_\infty$ (every pixel uses full budget)
- **JPEG re-encode q=85** — standard "imperceptible" reference (different budget; just for scale)

In [ ]:
import io
from PIL import Image

EPS = 8.0 / 255.0   # in [0, 1] scale

# Reuse `lfw_imgs`, `BATCH`, `N_TOTAL`, `psnr_fn`, `ssim_fn` from the previous cell.

psnr_u, ssim_u, linf_u = [], [], []
psnr_s, ssim_s, linf_s = [], [], []
psnr_j, ssim_j, linf_j = [], [], []

with torch.no_grad():
    for i in tqdm(range(0, N_TOTAL, BATCH), desc='LFW baselines'):
        x_raw = lfw_imgs[i:i + BATCH].to(DEVICE)         # [0, 255]
        x01   = x_raw / 255.0                             # [0, 1]

        # (a) Random uniform noise [-ε, ε]
        n_u = (torch.rand_like(x01) * 2 - 1) * EPS
        p_u = (x01 + n_u).clamp(0, 1)

        # (b) Random sign noise ±ε (worst-case for fixed L∞)
        n_s = (torch.empty_like(x01).bernoulli_(0.5) * 2 - 1) * EPS
        p_s = (x01 + n_s).clamp(0, 1)

        for p, lp, ls, ll in [(p_u, psnr_u, ssim_u, linf_u),
                              (p_s, psnr_s, ssim_s, linf_s)]:
            ll.append(((p - x01).abs().amax(dim=[1, 2, 3]) * 255.0).cpu().numpy())
            xn = x01.cpu().numpy().transpose(0, 2, 3, 1)
            pn = p.cpu().numpy().transpose(0, 2, 3, 1)
            for j in range(xn.shape[0]):
                lp.append(psnr_fn(xn[j], pn[j], data_range=1.0))
                ls.append(ssim_fn(xn[j], pn[j], data_range=1.0, channel_axis=-1))

        # (c) JPEG re-encode at quality 85 (different budget; reference only)
        x_uint = (x01 * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().transpose(0, 2, 3, 1)
        p_j_list = []
        for j in range(x_uint.shape[0]):
            buf = io.BytesIO()
            Image.fromarray(x_uint[j]).save(buf, format='JPEG', quality=85)
            buf.seek(0)
            p_j_list.append(np.array(Image.open(buf).convert('RGB'), dtype=np.float32) / 255.0)
        p_j = torch.from_numpy(np.stack(p_j_list).transpose(0, 3, 1, 2)).to(DEVICE)

        linf_j.append(((p_j - x01).abs().amax(dim=[1, 2, 3]) * 255.0).cpu().numpy())
        xn = x01.cpu().numpy().transpose(0, 2, 3, 1)
        pn = p_j.cpu().numpy().transpose(0, 2, 3, 1)
        for j in range(xn.shape[0]):
            psnr_j.append(psnr_fn(xn[j], pn[j], data_range=1.0))
            ssim_j.append(ssim_fn(xn[j], pn[j], data_range=1.0, channel_axis=-1))


def _report(name, p, s, l):
    p = np.array(p); l = np.concatenate(l); s = np.array(s)
    print(f'  {name:24s}: PSNR={p.mean():6.2f}±{p.std():.2f}  '
          f'SSIM={s.mean():.4f}±{s.std():.4f}  '
          f'L_inf(0-255)={l.mean():6.2f}±{l.std():.2f}')


print(f'\n[LFW SSIM sanity check | N={N_TOTAL}]')
_report('Random uniform [-ε,+ε]', psnr_u, ssim_u, linf_u)
_report('Random sign ±ε (worst)', psnr_s, ssim_s, linf_s)
_report('JPEG q=85 (diff budget)', psnr_j, ssim_j, linf_j)
print(f'  {"Our method":24s}: PSNR={psnrs.mean():6.2f}±{psnrs.std():.2f}  '
      f'SSIM={ssims.mean():.4f}±{ssims.std():.4f}  '
      f'L_inf(0-255)={linfs.mean():6.2f}±{linfs.std():.2f}')

In [ ]:
from torchmetrics.functional.image import structural_similarity_index_measure as tm_ssim

# Reuse `lfw_imgs`, `BATCH`, `N_TOTAL`, `G`, `tqdm` from earlier cells.

EPS = 8.0 / 255.0


@torch.no_grad()
def per_image_psnr(p, x, data_range=1.0):
    """Per-image PSNR over (C, H, W). Returns shape [B]."""
    mse = ((p - x) ** 2).mean(dim=[1, 2, 3])
    return 10.0 * torch.log10((data_range ** 2) / mse.clamp(min=1e-12))


ours_psnr, ours_ssim, ours_linf = [], [], []
unif_psnr, unif_ssim            = [], []
sign_psnr, sign_ssim            = [], []

G.eval()
with torch.no_grad():
    for i in tqdm(range(0, N_TOTAL, BATCH), desc='torchmetrics'):
        x_raw   = lfw_imgs[i:i + BATCH].to(DEVICE)
        x_batch = (x_raw - 127.5) / 127.5

        # --- Our method ---
        d_batch = G(x_batch)
        p_batch = (x_batch + d_batch).clamp(-1, 1)
        x01 = (x_batch + 1) / 2
        p01 = (p_batch + 1) / 2
        ours_psnr.append(per_image_psnr(p01, x01).cpu().numpy())
        ours_ssim.append(tm_ssim(p01, x01, data_range=1.0, reduction='none').cpu().numpy())
        ours_linf.append(((p01 - x01).abs().amax(dim=[1, 2, 3]) * 255.0).cpu().numpy())

        # --- Baseline: random uniform [-ε, ε] ---
        n_u = (torch.rand_like(x01) * 2 - 1) * EPS
        p_u = (x01 + n_u).clamp(0, 1)
        unif_psnr.append(per_image_psnr(p_u, x01).cpu().numpy())
        unif_ssim.append(tm_ssim(p_u, x01, data_range=1.0, reduction='none').cpu().numpy())

        # --- Baseline: random sign ±ε (worst case L∞) ---
        n_s = (torch.empty_like(x01).bernoulli_(0.5) * 2 - 1) * EPS
        p_s = (x01 + n_s).clamp(0, 1)
        sign_psnr.append(per_image_psnr(p_s, x01).cpu().numpy())
        sign_ssim.append(tm_ssim(p_s, x01, data_range=1.0, reduction='none').cpu().numpy())


def _show(name, p, s, l=None):
    p = np.concatenate(p); s = np.concatenate(s)
    msg = (f'  {name:24s}: PSNR={p.mean():6.2f}±{p.std():.2f}  '
           f'SSIM={s.mean():.4f}±{s.std():.4f}')
    if l is not None:
        l = np.concatenate(l)
        msg += f'  L_inf(0-255)={l.mean():6.2f}±{l.std():.2f}'
    print(msg)


print(f'\n[torchmetrics — Wang 11x11 Gaussian, sigma=1.5 | N={N_TOTAL}]')
_show('Our method',             ours_psnr, ours_ssim, ours_linf)
_show('Random uniform [-ε,+ε]', unif_psnr, unif_ssim)
_show('Random sign ±ε (worst)', sign_psnr, sign_ssim)